In [ ]:
import os
import shutil

import numpy as np
import pandas as pd

from guild.bulk import BulkRun

np.random.seed(42)
PROJECT_NAME = "vinarunknownbinders"

# Clean the folder but preserve known_binders.csv if it exists
project_dir = f"../../data/{PROJECT_NAME}"
known_binders_backup = f"/tmp/known_binders_{PROJECT_NAME}.csv"

# Backup known_binders.csv if it exists
if os.path.exists(f"{project_dir}/known_binders.csv"):
    shutil.copy(f"{project_dir}/known_binders.csv", known_binders_backup)
    print("✓ Backed up known_binders.csv")

# Remove the entire project directory
shutil.rmtree(project_dir, ignore_errors=True)

# Recreate directory structure
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/batches", exist_ok=True)
os.makedirs(f"{project_dir}/plots", exist_ok=True)

# Restore known_binders.csv if backup exists
if os.path.exists(known_binders_backup):
    shutil.copy(known_binders_backup, f"{project_dir}/known_binders.csv")
    print("✓ Restored known_binders.csv")
    os.remove(known_binders_backup)
else:
    print(f"⚠️ No known_binders.csv found - please paste your backup to: {os.path.abspath(project_dir)}/known_binders.csv")

In [ ]:
runs_table = pd.read_csv(
    "notebooks/data_prep/full_combinations_table.csv", sep="\t"
)


In [ ]:
# Load the known binders and merge with protein information from runs_table

import pandas as pd

project_dir = f"../../data/{PROJECT_NAME}"

# Load known binders CSV (has: uniprot_id, activity, chembl_id, smiles, pchembl_value, pdb_id)
kb = pd.read_csv(f"{project_dir}/known_binders.csv")

# Merge with runs_table to get protein paths and metadata
# Map pdb_id from known_binders to protein_id in runs_table
known_binders_only = kb.merge(
    runs_table[['protein_id', 'protein_config_id', 'protein_chain', 'protein_path', 
                'original_ligand', 'original_ligand_chain', 'is_pdb']].drop_duplicates(),
    left_on='pdb_id',
    right_on='protein_id',
    how='inner'
)

# Rename and format to match expected schema
known_binders_only = known_binders_only.rename(columns={
    'chembl_id': 'ligand_id',
    'activity': 'ligand_category'
})

print(f"Known binders table shape: {known_binders_only.shape}")
print(f"Unique proteins: {known_binders_only['protein_id'].nunique()}")
print(f"Columns: {known_binders_only.columns.tolist()}")
known_binders_only.head()

## Setup for Known Binders Only Run
We'll create a minimal protein-only table and let the known binders mechanism populate the ligands.

In [ ]:
# Optional: Verify the known_binders data is loaded correctly
print(f"✓ Loaded {len(known_binders_only)} known binder combinations")
print(f"✓ Proteins: {known_binders_only['protein_id'].unique()}")

In [ ]:
# Now pass ONLY the known binders table (no placeholders, no decoys)
bulk_analysis_object = BulkRun(
    known_binders_only,  # Pure known binders table
    PROJECT_NAME,
    methods_to_run=["vina"],
    batch_size=50,
    use_known_binders=False,  # Already have them, don't fetch again
    use_decoys=False,
    n_workers=1,
)

In [ ]:
# Debug: Check the known binders
print("All combinations shape:", bulk_analysis_object.all_combinations_table.shape)
print("\nKnown binders shape:", bulk_analysis_object.known_binders_table.shape if hasattr(bulk_analysis_object, 'known_binders_table') else "Not set")

if hasattr(bulk_analysis_object, 'known_binders_table') and not bulk_analysis_object.known_binders_table.empty:
    print("\nUnique proteins with known binders:")
    print(bulk_analysis_object.known_binders_table['protein_id'].value_counts())
    print("\nFirst few known binder rows:")
    print(bulk_analysis_object.known_binders_table.head())
else:
    print("\nNo known binders table generated")

print("\nLigand categories in all combinations:")
print(bulk_analysis_object.all_combinations_table['ligand_category'].value_counts())

In [ ]:
bulk_analysis_object.run_docking()

In [ ]:
bulk_analysis_object.run_guild_scoring()

In [ ]:
bulk_analysis_object.run_interactions_analysis()